# 04_03 Choosing a classifier: which one, for which job?

You now have three families of classifier (nearest neighbours, Naive Bayes, and Lab 03's logistic regression)
and two kinds of features (word counts and sentence embeddings). This notebook puts five combinations on the
same split, measures accuracy **and** cost, and asks you to choose one for each of three jobs.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-04-which-classifier-and-why", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'sentence_transformers': 'sentence-transformers',
           'sklearn': 'scikit-learn',
           'pandas': 'pandas',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
import clftools
from nlpcheck import ask, guess, reveal, check_04_03

X_train, X_test, y_train, y_test = clftools.split(clftools.load_sentences())

## 1. Recall

**r5.** Why did Naive Bayes call "not bad" 99 percent negative? (a) it treats "not" and "bad" as independent
negative evidence, (b) it removed stop words, (c) smoothing

**r6.** What does Laplace smoothing do? (a) removes rare words, (b) normalises the counts, (c) adds one to every
count so an unseen word does not make the product zero

In [ ]:
ask("r5", "")
ask("r6", "")

## 2. Five models, one split

The sentence embeddings below come from `clftools.embed()`. Your session started computing them in the
background the moment it began, while you read the chapter, and stores each one under `out/.embeddings/`.
That takes about four minutes. If you got here sooner, this cell finishes the job itself, which can take up to
about six minutes on a session's computer if none of it was done; the line it prints says how many vectors were
already waiting. Once they are all cached, the whole notebook runs in well under half a minute.

In [ ]:
t = time.time()
E_train, E_test = clftools.embed(X_train), clftools.embed(X_test)
print(f"embeddings ready in {time.time() - t:.1f} s")

models = {
    "naive_bayes":  (make_pipeline(CountVectorizer(), MultinomialNB()), X_train, X_test),
    "knn_tfidf":    (make_pipeline(TfidfVectorizer(), KNeighborsClassifier(n_neighbors=25, metric="cosine")), X_train, X_test),
    "logreg_tfidf": (make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000)), X_train, X_test),
    "knn_embed":    (KNeighborsClassifier(n_neighbors=51, metric="cosine"), E_train, E_test),
    "logreg_embed": (LogisticRegression(max_iter=2000, C=10), E_train, E_test),
}
guess("most_accurate", None)   # which of the five names above do you expect to be most accurate?

In [ ]:
results = {}
for name, (model, Xa, Xb) in models.items():
    t = time.time(); model.fit(Xa, y_train); fit_s = time.time() - t
    t = time.time(); acc = model.score(Xb, y_test); pred_s = time.time() - t
    results[name] = {"accuracy": round(acc, 4), "fit_ms": round(fit_s * 1000, 1), "predict_ms_per_1000": round(pred_s / len(y_test) * 1e6, 1)}
table = pd.DataFrame(results).T.sort_values("accuracy", ascending=False)
print(table)
reveal("most_accurate", table.index[0])

Logistic regression on embeddings is the most accurate, about 0.84, and the gap is the point of section 5 of the
first notebook. KNN on the same embeddings scores about 0.80, because it lets every direction of meaning count
equally; logistic regression **learns** which directions matter for sentiment and weights them. The embeddings
carry the information. The trained model finds it.

Read the cost columns too. The embedding models' timings leave out the minutes it took to embed the sentences in
the first place; on the right-hand side, only Naive Bayes and logistic regression on counts can handle millions
of messages a minute on one machine. KNN is cheap to fit (it only stores) and pays at prediction time instead,
comparing each new sentence with every stored one.

## 3. What each gets wrong

Accuracy hides which sentences a model fails on. These are test sentences where the best and the cheapest
models disagree:

In [ ]:
nb_pred = models["naive_bayes"][0].predict(X_test)
lr_pred = models["logreg_embed"][0].predict(E_test)
disagree = pd.DataFrame({"text": X_test.values, "true": y_test.values, "naive_bayes": nb_pred, "logreg_embed": lr_pred})
disagree = disagree[disagree.naive_bayes != disagree.logreg_embed]
print(len(disagree), "disagreements")
disagree.head(8)

Read a few. Where the embedding model wins, the sentence usually says something positive without a positive
word, or is sarcastic, or uses a word Naive Bayes never saw; where Naive Bayes wins, a single strong word
decided it. No model is simply better: they fail on different sentences.

## 4. Your turn: choose for the job

Fill in `choices` with one of the five names for each job, using your table above:

- `cheapest`: a filter that must classify ten million messages a day on one small machine;
- `most_accurate`: a monthly report where accuracy is all that matters;
- `no_retraining`: a support tool where agents add new labelled examples all day, and each should count at once,
  with no retraining.

In [ ]:
choices = {
    "cheapest": None,        # YOUR CODE HERE
    "most_accurate": None,   # YOUR CODE HERE
    "no_retraining": None,   # YOUR CODE HERE
}
os.makedirs("out", exist_ok=True)
json.dump({"results": results, "choices": choices}, open("out/04_03_choice.json", "w"), indent=1)
check_04_03()

## 5. Exit ticket

**x4.** True or false: KNN does more computation at test time than at training time. (a) true, (b) false

In [ ]:
ask("x4", "")

Explain it back: the embedding models were the most accurate. Name one situation where you would still ship
Naive Bayes.

*Your explanation:* 